In [216]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
from numpy import random
from collections import Counter
from numpy.linalg import inv
from numpy.linalg import eig
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import pylab
from pylab import legend
from pylab import plot, show, title, xlabel, ylabel
import scipy
from scipy import stats
from scipy.stats import binom,poisson,norm,t,expon,f
from sklearn.model_selection import train_test_split

import statsmodels
from statsmodels import stats
from statsmodels.stats import weightstats as ssw

import statsmodels.api as sm
from statsmodels.formula.api import ols
import statsmodels.stats.multicomp

import statsmodels.stats
from statsmodels.stats.proportion import proportions_ztest


from scipy.stats import chisquare

from  scipy.stats import chi2_contingency

# 3.Paired T-test

In [54]:
random.seed(10)
before=np.random.randint(10,30,7)
after=np.random.randint(15,30,7)
print(before,after)

[19 14 25 10 27 26 27] [25 23 24 15 25 23 21]


In [55]:
# Ho: avg(after-before)>=2
# Ha: avg(before-after)<2 ---------->Left tail Test
d=(after-before)
d

array([ 6,  9, -1,  5, -2, -3, -6], dtype=int32)

In [56]:
np.mean(d)

np.float64(1.1428571428571428)

In [57]:
ssw.ttost_paired(before,after,low=2,upp=2) #1st way to find Paired t-test

(np.float64(0.9086396247301425),
 (np.float64(-1.5062370331392065),
  np.float64(0.9086396247301425),
  np.float64(6.0)),
 (np.float64(-1.5062370331392065),
  np.float64(0.09136037526985749),
  np.float64(6.0)))

In [68]:
scipy.stats.ttest_1samp(before-after,popmean=2,alternative="greater") # 2nd way to find the paired t-test

TtestResult(statistic=np.float64(-1.5062370331392065), pvalue=np.float64(0.9086396247301425), df=np.int64(6))

In [69]:
#Test Statistics
(np.mean(d)-2)/((np.std(d,ddof=1))/(6**0.5))

np.float64(-0.3803194146278324)

In [70]:
t.cdf(-0.6726442943930657,5)

np.float64(0.2655009101029434)

# 4.One Way ANOVA(Analysis of Variance)

In [64]:
random.seed(42)
dfA = pd.DataFrame({'Col1':np.repeat('A',7),'Col2':np.random.randint(10,30,7)})
dfB = pd.DataFrame({'Col1':np.repeat('B',9),'Col2':np.random.randint(20,40,9)})
dfC = pd.DataFrame({'Col1':np.repeat('C',8),'Col2':np.random.randint(15,30,8)})
df = pd.concat([dfA,dfB,dfC])

In [65]:
df

,Col1,Col2
0,A,16
1,A,29
2,A,24
3,A,20
4,A,17
5,A,16
6,A,28
0,B,30
1,B,30
2,B,23


In [71]:
mod1=ols("Col2 ~ Col1",df).fit()

In [73]:
tbl=sm.stats.anova_lm(mod1)

In [86]:
tbl

,df,sum_sq,mean_sq,F,PR(>F)
Col1,2.0,71.146825,35.573413,1.649785,0.216032
Residual,21.0,452.811508,21.562453,NaN,NaN


In [87]:
print(sm.stats.multicomp.pairwise_tukeyhsd(df.Col2,df.Col1))

Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
     A      B    4.127 0.2061 -1.7715 10.0254  False
     A      C   1.4464 0.8206 -4.6112   7.504  False
     B      C  -2.6806 0.4731 -8.3679  3.0067  False
----------------------------------------------------


In [119]:
norm.cdf()

480

## ANOVA Example


In [100]:
df = pd.read_excel('data_anova.xlsx')
df.index = np.arange(df.shape[0])

In [101]:
df

,Col1,Col2
0,A,10
1,A,23
2,A,14
3,A,25
4,A,22
5,A,11
6,A,12
7,B,26
8,B,39
9,B,29


In [102]:
grp1 = df.groupby('Col1')

In [103]:
samp_mean = grp1.agg('mean')
grand_mean = np.mean(df.Col2)


In [104]:
7*(samp_mean.loc['A'] - grand_mean)**2 + 9*(samp_mean.loc['B'] - grand_mean)**2 + 8*(samp_mean.loc['C'] - grand_mean)**2


Col2    842.849206
dtype: float64

In [105]:
sampA = df.Col2[np.where(df.Col1=='A')[0]]
sampB = df.Col2[np.where(df.Col1=='B')[0]]
sampC = df.Col2[np.where(df.Col1=='C')[0]]

In [106]:
sampA,sampB,sampC

(0    10
 1    23
 2    14
 3    25
 4    22
 5    11
 6    12
 Name: Col2, dtype: int64,
 7     26
 8     39
 9     29
 10    39
 11    31
 12    32
 13    38
 14    22
 15    25
 Name: Col2, dtype: int64,
 16    21
 17    23
 18    26
 19    19
 20    29
 21    27
 22    17
 23    24
 Name: Col2, dtype: int64)

In [107]:
sum((np.array(sampA) - np.array(samp_mean.loc['A']))**2) + sum((np.array(sampB) - np.array(samp_mean.loc['B']))**2) + sum((np.array(sampC) - np.array(samp_mean.loc['C']))**2)

np.float64(684.484126984127)

In [114]:
842.84/684.48

1.2313581112669472

In [118]:
684/21

32.57142857142857

# 5.Two Way ANOVA


In [129]:
zoo = [34,43,57,40,85,68,67,53,41,24,42,52]
supp=[1,1,1,1,2,2,2,2,3,3,3,3]
lake=["R","D","R","D","R","D","R","D","R","D","R","D"]
df=pd.DataFrame({'Resp':zoo,"Factor1":supp,"Factor2":lake})

In [130]:
df

,Resp,Factor1,Factor2
0,34,1,R
1,43,1,D
2,57,1,R
3,40,1,D
4,85,2,R
5,68,2,D
6,67,2,R
7,53,2,D
8,41,3,R
9,24,3,D


In [138]:
model=ols("Resp~Factor1*Factor2",df).fit()

In [139]:
res=sm.stats.anova_lm(model)

In [140]:
res

,df,sum_sq,mean_sq,F,PR(>F)
Factor2,1.0,176.333333,176.333333,0.483367,0.506590
Factor1,1.0,28.125000,28.125000,0.077097,0.788312
Factor1:Factor2,1.0,0.125000,0.125000,0.000343,0.985685
Residual,8.0,2918.416667,364.802083,NaN,NaN


# 6.One Proportion Test

In [148]:
mu=500*0.7
mu

350.0

In [150]:
sd=(500*0.7*0.3)**0.5
sd

10.246950765959598

In [153]:
binom.cdf(370,500,0.7)- binom.cdf(329,500,0.7) # Porbobability between 330 and 370

np.float64(0.9547133734363867)

In [155]:
norm.cdf(370.5,mu,sd)-norm.cdf(329.5,mu,sd)

np.float64(0.9545639632657537)

In [156]:
#Ho: pr<=0.4   ,   Ha: pr>0.4

In [157]:
# IN Sample of 400 students , 180 students are from MH

In [158]:
180/400

0.45

In [166]:
statsmodels.stats.proportion.proportions_ztest(180,400,0.4,alternative="larger")

(np.float64(2.0100756305184238), np.float64(0.022211592425003045))

In [167]:
norm.cdf(2.0100756305184238)

np.float64(0.977788407574997)

In [168]:
1-norm.cdf(2.0100756305184238)

np.float64(0.022211592425003035)

# 7.Two Proportion Test

In [173]:
#  Pune Cdac claims that they have alteast 8% engineer as compared to CDAC khaghar
# Ho:pr(pune)-pr(khar)>=40
# Ha:pr(pune)-pr(khar)<40

# 70 student / 40 engineer (Pune)
# 80  Students /48 engineer (Khar)


In [174]:
engg=np.array([40,48])
total=np.array([70,80])

In [175]:
statsmodels.stats.proportion.proportions_ztest(engg,total,0.08,alternative="smaller")

(np.float64(-1.3471563869719667), np.float64(0.08896493545619544))

# 8.Chi Square Test for Goodness of Fit

In [178]:
# Claimed pattern for the qualifications
# btech -0.4 ,Mtech-0.1,MCA-0.3,Msc- 0.2
# btech - 50 ,Mtech -20, MCA -45, Msc=25
# whether counts obtains from  sample follow the pattern or not

In [207]:
counts=np.array([50,20,45,35])

In [208]:
pattern=np.array([0.4,0.1,0.3,0.2])

In [209]:
exp_count= sum(counts) *pattern
exp_count

array([60., 15., 45., 30.])

In [187]:
 scipy.stats.chisquare(counts,f_exp=exp_count)

Power_divergenceResult(statistic=np.float64(4.166666666666667), pvalue=np.float64(0.24402044505670617))

In [211]:
counts=np.array([70,10,35,45])

In [212]:
exp_count= sum(counts) *pattern
exp_count

array([64., 16., 48., 32.])

In [213]:
 scipy.stats.chisquare(counts,f_exp=exp_count)

Power_divergenceResult(statistic=np.float64(11.614583333333334), pvalue=np.float64(0.008827096989130126))

In [214]:
sum((exp_count-counts)**2/exp_count)

np.float64(11.614583333333334)

In [215]:
counts = np.array([70,10,35,45])
exp_counts = sum(counts) * pattern
scipy.stats.chisquare(counts, f_exp = exp_counts)
# p-value<0.05, so we reject Ho

exp_counts
# Out[60]: array([64., 16., 48., 32.])

sum((exp_counts - counts)**2/exp_counts)
# Out[61]: np.float64(11.614583333333334)

(exp_counts - counts)**2/exp_counts
# Out[62]: array([0.5625    , 2.25      , 3.52083333, 5.28125   ])

array([0.5625    , 2.25      , 3.52083333, 5.28125   ])

# 9.Chi Square test for degree of association


In [225]:
#Ho: Choice of food does not depend on Gender
#Ha: Choice of food depends on Gender


In [222]:
food=np.array([[45,70,60],[70,40,25]])

In [223]:
food

array([[45, 70, 60],
       [70, 40, 25]])

In [224]:
scipy.stats.chi2_contingency(food)

Chi2ContingencyResult(statistic=np.float64(23.25424258416586), pvalue=np.float64(8.920832446902749e-06), dof=2, expected_freq=array([[64.91935484, 62.09677419, 47.98387097],
       [50.08064516, 47.90322581, 37.01612903]]))